In [1]:
import os
os.chdir("D:/Projects/volatility-radar")

In [2]:
import pandas as pd

df = pd.read_csv("data/final_cleaned_data/full_features.csv")
df.shape

(4023, 40)

In [3]:
df['date'] = pd.to_datetime(df['date'])
unique_dates = df['date'].unique()

print(unique_dates[0])
print(unique_dates[-1])

2021-01-21 00:00:00
2026-03-12 00:00:00


In [19]:
def walk_forward_splits(unique_dates, initial_train_months=24, test_months=3, gap_days=14):
    splits = []
    unique_dates = pd.Series(unique_dates).sort_values().reset_index(drop=True)
    
    first_train_start = unique_dates.iloc[0]
    first_test_start = first_train_start + pd.DateOffset(months=initial_train_months)
    
    train_start = first_train_start
    test_start = first_test_start
    
    while True:
        test_end = test_start + pd.DateOffset(months=test_months)
        
        # stop if test window exceeds available data
        if test_end > unique_dates.iloc[-1]:
            break
        
        train_end = test_start - pd.Timedelta(days=gap_days)
        
        train_dates = unique_dates[(unique_dates >= train_start) & (unique_dates <= train_end)]
        test_dates = unique_dates[(unique_dates >= test_start) & (unique_dates <= test_end)]
        
        # only add fold if both windows have data
        if len(train_dates) > 0 and len(test_dates) > 0:
            splits.append((train_dates.values, test_dates.values))
        
        test_start = test_start + pd.DateOffset(months=test_months)

        train_start = train_start + pd.DateOffset(months=test_months)
    
    return splits

In [21]:
splits = walk_forward_splits(unique_dates)
print(f"Total folds: {len(splits)}")
for i, (train, test) in enumerate(splits):
    print(f"Fold {i+1}: Train {train[0].astype('datetime64[D]')} → {train[-1].astype('datetime64[D]')} ({len(train)} days) | Test {test[0].astype('datetime64[D]')} → {test[-1].astype('datetime64[D]')} ({len(test)} days)")

Total folds: 12
Fold 1: Train 2021-01-21 → 2023-01-06 (512 days) | Test 2023-01-23 → 2023-04-21 (65 days)
Fold 2: Train 2021-04-21 → 2023-04-07 (513 days) | Test 2023-04-21 → 2023-07-21 (66 days)
Fold 3: Train 2021-07-21 → 2023-07-07 (513 days) | Test 2023-07-21 → 2023-10-20 (66 days)
Fold 4: Train 2021-10-21 → 2023-10-06 (512 days) | Test 2023-10-23 → 2024-01-19 (65 days)
Fold 5: Train 2022-01-21 → 2024-01-05 (511 days) | Test 2024-01-22 → 2024-04-19 (65 days)
Fold 6: Train 2022-04-21 → 2024-04-05 (512 days) | Test 2024-04-22 → 2024-07-19 (65 days)
Fold 7: Train 2022-07-21 → 2024-07-05 (512 days) | Test 2024-07-22 → 2024-10-21 (66 days)
Fold 8: Train 2022-10-21 → 2024-10-07 (512 days) | Test 2024-10-21 → 2025-01-21 (67 days)
Fold 9: Train 2023-01-23 → 2025-01-07 (512 days) | Test 2025-01-21 → 2025-04-21 (65 days)
Fold 10: Train 2023-04-21 → 2025-04-07 (512 days) | Test 2025-04-21 → 2025-07-21 (66 days)
Fold 11: Train 2023-07-21 → 2025-07-07 (512 days) | Test 2025-07-21 → 2025-10-21 (6

In [22]:
splits = walk_forward_splits(unique_dates)
print(len(splits))
print(splits[0][0][0])
print(splits[0][1][0])
print(splits[1][0][0])

12
2021-01-21T00:00:00.000000
2023-01-23T00:00:00.000000
2021-04-21T00:00:00.000000


In [23]:
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, classification_report
import numpy as np

FEATURE_COLS = [
    'pair', 'open', 'high', 'low', 'close', 'day_of_week', 'month', 'week_of_year', 'is_month_end', 'is_month_start',
    'max_up_pips', 'max_down_pips', 'max_profit', 'max_loss', 'daily_return',
    'return_3d', 'return_5d', 'return_10d', 'rolling_std_5', 'rolling_std_10',
    'rolling_std_20', 'rsi_14', 'atr_14', 'momentum_5d', 'momentum_10d',
    'dist_from_mean_20d', 'daily_range', 'candle_body', 'upper_wick', 'lower_wick',
    'high_impact_count', 'medium_impact_count', 'low_impact_count', 'max_z_score',
    'sum_signal', 'dominant_direction', 'max_surprise_z', 'sum_signal_surprise'
]

TARGET_COL = 'label'

def run_walk_forward(df, splits):
    fold_results = []

    for i, (train_dates, test_dates) in enumerate(splits):
        train_df = df[df['date'].isin(train_dates)].copy()
        test_df = df[df['date'].isin(test_dates)].copy()

        X_train = train_df[FEATURE_COLS]
        y_train = train_df[TARGET_COL]
        X_test = test_df[FEATURE_COLS]
        y_test = test_df[TARGET_COL]

        model = XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric='mlogloss',
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        weighted_f1 = f1_score(y_test, y_pred, average='weighted')
        report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

        fold_results.append({
            'fold': i + 1,
            'train_size': len(train_df),
            'test_size': len(test_df),
            'weighted_f1': weighted_f1,
            'report': report,
            'y_test': y_test.values,
            'y_pred': y_pred
        })

        print(f"Fold {i+1} | Train: {len(train_df)} rows | Test: {len(test_df)} rows | Weighted F1: {weighted_f1:.4f}")

    return fold_results

fold_results = run_walk_forward(df, splits)

Fold 1 | Train: 1536 rows | Test: 195 rows | Weighted F1: 0.3717
Fold 2 | Train: 1539 rows | Test: 198 rows | Weighted F1: 0.3300
Fold 3 | Train: 1539 rows | Test: 198 rows | Weighted F1: 0.3806
Fold 4 | Train: 1536 rows | Test: 195 rows | Weighted F1: 0.3140
Fold 5 | Train: 1533 rows | Test: 195 rows | Weighted F1: 0.3444
Fold 6 | Train: 1536 rows | Test: 195 rows | Weighted F1: 0.3957
Fold 7 | Train: 1536 rows | Test: 198 rows | Weighted F1: 0.3218
Fold 8 | Train: 1536 rows | Test: 201 rows | Weighted F1: 0.3468
Fold 9 | Train: 1536 rows | Test: 195 rows | Weighted F1: 0.3274
Fold 10 | Train: 1536 rows | Test: 198 rows | Weighted F1: 0.3655
Fold 11 | Train: 1536 rows | Test: 201 rows | Weighted F1: 0.2970
Fold 12 | Train: 1536 rows | Test: 201 rows | Weighted F1: 0.2989


In [24]:
from sklearn.metrics import confusion_matrix
import numpy as np

all_y_test = np.concatenate([r['y_test'] for r in fold_results])
all_y_pred = np.concatenate([r['y_pred'] for r in fold_results])

print("Aggregated confusion matrix (all 12 folds):")
print(confusion_matrix(all_y_test, all_y_pred))
print("\nRows = actual, Columns = predicted")
print("Classes: 0=Bearish, 1=Neutral, 2=Bullish")

from sklearn.metrics import classification_report
print(classification_report(all_y_test, all_y_pred, target_names=['Bearish','Neutral','Bullish']))

Aggregated confusion matrix (all 12 folds):
[[336 162 291]
 [285 140 272]
 [326 201 357]]

Rows = actual, Columns = predicted
Classes: 0=Bearish, 1=Neutral, 2=Bullish
              precision    recall  f1-score   support

     Bearish       0.35      0.43      0.39       789
     Neutral       0.28      0.20      0.23       697
     Bullish       0.39      0.40      0.40       884

    accuracy                           0.35      2370
   macro avg       0.34      0.34      0.34      2370
weighted avg       0.34      0.35      0.35      2370



In [9]:
from sklearn.metrics import confusion_matrix
import numpy as np

all_y_test = np.concatenate([r['y_test'] for r in fold_results])
all_y_pred = np.concatenate([r['y_pred'] for r in fold_results])

print("Aggregated confusion matrix (all 12 folds):")
print(confusion_matrix(all_y_test, all_y_pred))
print("\nRows = actual, Columns = predicted")
print("Classes: 0=Bearish, 1=Neutral, 2=Bullish")

from sklearn.metrics import classification_report
print(classification_report(all_y_test, all_y_pred, target_names=['Bearish','Neutral','Bullish']))

Aggregated confusion matrix (all 12 folds):
[[362 148 279]
 [301 149 247]
 [359 193 332]]

Rows = actual, Columns = predicted
Classes: 0=Bearish, 1=Neutral, 2=Bullish
              precision    recall  f1-score   support

     Bearish       0.35      0.46      0.40       789
     Neutral       0.30      0.21      0.25       697
     Bullish       0.39      0.38      0.38       884

    accuracy                           0.36      2370
   macro avg       0.35      0.35      0.34      2370
weighted avg       0.35      0.36      0.35      2370



In [26]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'eval_metric': 'mlogloss',
        'random_state': 42,
        'n_jobs': -1
    }

    # use only first 6 folds for tuning — saves time
    fold_f1s = []
    for i, (train_dates, test_dates) in enumerate(splits[:6]):
        train_df = df[df['date'].isin(train_dates)]
        test_df = df[df['date'].isin(test_dates)]

        X_train, y_train = train_df[FEATURE_COLS], train_df[TARGET_COL]
        X_test, y_test = test_df[FEATURE_COLS], test_df[TARGET_COL]

        model = XGBClassifier(**params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        fold_f1s.append(f1_score(y_test, y_pred, average='weighted'))

    return np.mean(fold_f1s)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\nBest weighted F1: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

  0%|          | 0/50 [00:00<?, ?it/s]

[W 2026-06-03 11:38:03,050] Trial 10 failed with parameters: {'n_estimators': 213, 'max_depth': 5, 'learning_rate': 0.028740075623087894, 'subsample': 0.6105609506571645, 'colsample_bytree': 0.756401666853837, 'min_child_weight': 1, 'gamma': 0.042425270459438835} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "d:\projects\volatility-radar\.venv\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\simon\AppData\Local\Temp\ipykernel_22180\3605008148.py", line 28, in objective
    model.fit(X_train, y_train)
  File "d:\projects\volatility-radar\.venv\Lib\site-packages\xgboost\core.py", line 751, in inner_f
    return func(**kwargs)
           ^^^^^^^^^^^^^^
  File "d:\projects\volatility-radar\.venv\Lib\site-packages\xgboost\sklearn.py", line 1806, in fit
    self._Booster = train(
                    ^^^^^^
  File "d:\projects\volatility-ra

KeyboardInterrupt: 

In [11]:
print(df['label'].value_counts(normalize=True).sort_index())

label
0.0    0.334328
1.0    0.297539
2.0    0.368133
Name: proportion, dtype: float64


In [12]:
PAIRS = {0: 'EURUSD', 1: 'GBPUSD', 2: 'USDJPY'}

per_pair_results = {}

for pair_id, pair_name in PAIRS.items():
    print(f"\n--- {pair_name} ---")
    pair_df = df[df['pair'] == pair_id].copy()
    
    pair_unique_dates = pair_df['date'].sort_values().unique()
    pair_splits = walk_forward_splits(pair_unique_dates)
    
    # drop pair column — not needed when training per pair
    pair_feature_cols = [c for c in FEATURE_COLS if c != 'pair']
    
    fold_f1s = []
    fold_preds = []
    
    for i, (train_dates, test_dates) in enumerate(pair_splits):
        train_df = pair_df[pair_df['date'].isin(train_dates)]
        test_df = pair_df[pair_df['date'].isin(test_dates)]
        
        X_train, y_train = train_df[pair_feature_cols], train_df[TARGET_COL]
        X_test, y_test = test_df[pair_feature_cols], test_df[TARGET_COL]
        
        model = XGBClassifier(
            n_estimators=351,
            max_depth=3,
            learning_rate=0.11843497553792567,
            subsample=0.7391796008578488,
            colsample_bytree=0.7829886570524643,
            min_child_weight=9,
            gamma=1.2278023664651414,
            eval_metric='mlogloss',
            random_state=42,
            n_jobs=-1
        )
        

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        wf1 = f1_score(y_test, y_pred, average='weighted')
        fold_f1s.append(wf1)
        fold_preds.append((y_test.values, y_pred))
        
        print(f"Fold {i+1} | F1: {wf1:.4f}")
    
    avg_f1 = np.mean(fold_f1s)
    print(f"{pair_name} Average F1: {avg_f1:.4f}")
    per_pair_results[pair_name] = {'fold_f1s': fold_f1s, 'avg_f1': avg_f1, 'fold_preds': fold_preds}


--- EURUSD ---
Fold 1 | F1: 0.3310
Fold 2 | F1: 0.4073
Fold 3 | F1: 0.3463
Fold 4 | F1: 0.2897
Fold 5 | F1: 0.3375
Fold 6 | F1: 0.3402
Fold 7 | F1: 0.3281
Fold 8 | F1: 0.2073
Fold 9 | F1: 0.4409
Fold 10 | F1: 0.2787
Fold 11 | F1: 0.3146
Fold 12 | F1: 0.2085
EURUSD Average F1: 0.3192

--- GBPUSD ---
Fold 1 | F1: 0.4009
Fold 2 | F1: 0.3700
Fold 3 | F1: 0.2906
Fold 4 | F1: 0.3908
Fold 5 | F1: 0.3084
Fold 6 | F1: 0.3753
Fold 7 | F1: 0.3649
Fold 8 | F1: 0.3486
Fold 9 | F1: 0.3306
Fold 10 | F1: 0.3415
Fold 11 | F1: 0.3657
Fold 12 | F1: 0.2053
GBPUSD Average F1: 0.3411

--- USDJPY ---
Fold 1 | F1: 0.3818
Fold 2 | F1: 0.3183
Fold 3 | F1: 0.4198
Fold 4 | F1: 0.3091
Fold 5 | F1: 0.2512
Fold 6 | F1: 0.4055
Fold 7 | F1: 0.3893
Fold 8 | F1: 0.2923
Fold 9 | F1: 0.2962
Fold 10 | F1: 0.3423
Fold 11 | F1: 0.3218
Fold 12 | F1: 0.3252
USDJPY Average F1: 0.3377


In [10]:
import joblib
import os
from xgboost import XGBClassifier

FEATURE_COLS = [
    'pair', 'day_of_week', 'month', 'week_of_year', 'is_month_end', 'is_month_start',
    'max_up_pips', 'max_down_pips', 'max_profit', 'max_loss', 'daily_return',
    'return_3d', 'return_5d', 'return_10d', 'rolling_std_5', 'rolling_std_10',
    'rolling_std_20', 'rsi_14', 'atr_14', 'momentum_5d', 'momentum_10d',
    'dist_from_mean_20d', 'daily_range', 'candle_body', 'upper_wick', 'lower_wick',
    'high_impact_count', 'medium_impact_count', 'low_impact_count', 'max_z_score',
    'sum_signal', 'dominant_direction', 'max_surprise_z', 'sum_signal_surprise'
]

TARGET_COL = 'label'

final_model = XGBClassifier(
            n_estimators=351,
            max_depth=3,
            learning_rate=0.11843497553792567,
            subsample=0.7391796008578488,
            colsample_bytree=0.7829886570524643,
            min_child_weight=9,
            gamma=1.2278023664651414,
            eval_metric='mlogloss',
            random_state=42,
            n_jobs=-1
        )

X_full = df[FEATURE_COLS]
y_full = df[TARGET_COL]

final_model.fit(X_full, y_full)

os.makedirs('src/models', exist_ok=True)
joblib.dump(final_model, 'src/models/volatility_radar_xgb.pkl')
print("Model saved.")

Model saved.


In [11]:
from dotenv import load_dotenv
load_dotenv(dotenv_path='D:/Projects/volatility-radar/.env')

True

In [3]:
import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
print("KEY:", api_key[:8] if api_key else "NONE")  # prints first 8 chars only
client = OpenAI(api_key=api_key)

KEY: sk-proj-
